# Step 1: Data Download & Initial Exploration
**INT 396 — Unsupervised Learning | Social Media Topic Discovery for Disaster Response**

This notebook downloads the HumAID and CrisisBench datasets from HuggingFace and saves them locally.

In [1]:
# Install dependencies (run once)\n!pip install -q datasets sentence-transformers umap-learn hdbscan wordcloud streamlit pyyaml pyarrow

In [2]:
import os
from pathlib import Path

import pandas as pd
from datasets import load_dataset

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "requirements.txt").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
PROJECT_ROOT = str(PROJECT_ROOT)

# Create directories
os.makedirs(f"{PROJECT_ROOT}/data/raw", exist_ok=True)
os.makedirs(f"{PROJECT_ROOT}/data/processed", exist_ok=True)
os.makedirs(f"{PROJECT_ROOT}/data/embeddings", exist_ok=True)
os.makedirs(f"{PROJECT_ROOT}/results", exist_ok=True)
os.makedirs(f"{PROJECT_ROOT}/figures", exist_ok=True)

/Users/cyril/Desktop/Un-supervised/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1.1 Download HumAID (Primary Dataset — 76,484 tweets)

In [3]:
# Download HumAID from HuggingFace
# The dataset has a metadata bug (expects 'dev' split but data has 'validation')
# So we skip verification checks

# Clear any cached broken download first
import shutil, glob
for p in glob.glob(os.path.expanduser("~/.cache/huggingface/datasets/QCRI___hum_aid*")):
    shutil.rmtree(p, ignore_errors=True)
for p in glob.glob(os.path.expanduser("~/.cache/huggingface/datasets/QCRI___HumAID*")):
    shutil.rmtree(p, ignore_errors=True)

humaid = load_dataset("QCRI/HumAID-all", verification_mode="no_checks")
print(f"HumAID splits: {list(humaid.keys())}")

# Combine all splits into one DataFrame
dfs = []
for split_name in humaid:
    df = humaid[split_name].to_pandas()
    df["split"] = split_name
    dfs.append(df)

humaid_df = pd.concat(dfs, ignore_index=True)
print(f"Total rows: {len(humaid_df):,}")
print(f"Columns: {list(humaid_df.columns)}")
humaid_df.head()

Generating train split: 100%|██████████| 53531/53531 [00:00<00:00, 3249656.80 examples/s]
Generating validation split: 7793 examples [00:00, 1464304.77 examples/s]
Generating test split: 100%|██████████| 15160/15160 [00:00<00:00, 2439315.94 examples/s]

HumAID splits: ['train', 'validation', 'test']
Total rows: 76,484
Columns: ['tweet_text', 'class_label', 'split']


,tweet_text,class_label,split
0,Powerful Ecuador quake kills at least 235: POR...,injured_or_dead_people,train
1,Im at awe and saddened with the #EcuadorEarthq...,rescue_volunteering_or_donation_effort,train
2,RT @RachelAndJun: Our hearts are with everyone...,sympathy_and_support,train
3,RT @noticias2000: Ecuador quake death toll has...,injured_or_dead_people,train
4,RT @pzf: BREAKING PHOTOS: Major damage reporte...,infrastructure_and_utility_damage,train


In [4]:
# Quick stats
print(f"Missing text values: {humaid_df['tweet_text'].isna().sum()}")
print(f"Average tweet length: {humaid_df['tweet_text'].str.len().mean():.0f} chars")
print(f"\nLabel distribution:")
print(humaid_df["class_label"].value_counts())

Missing text values: 0
Average tweet length: 142 chars

Label distribution:
class_label
rescue_volunteering_or_donation_effort    21278
other_relevant_information                12144
sympathy_and_support                       8931
infrastructure_and_utility_damage          8163
injured_or_dead_people                     7303
not_humanitarian                           6296
caution_and_advice                         5394
displaced_people_and_evacuations           3999
requests_or_urgent_needs                   2618
missing_or_found_people                     358
Name: count, dtype: int64


In [5]:
# Save raw HumAID
humaid_df.to_parquet(f"{PROJECT_ROOT}/data/raw/humaid_raw.parquet", index=False)
print(f"Saved: {PROJECT_ROOT}/data/raw/humaid_raw.parquet ({len(humaid_df):,} rows)")

Saved: /Users/cyril/Desktop/Un-supervised/data/raw/humaid_raw.parquet (76,484 rows)


## 1.2 Download CrisisBench (Validation Dataset — 132,619 tweets)

In [6]:
# Download CrisisBench (humanitarian config)
crisisbench = load_dataset("QCRI/CrisisBench-all-lang", "humanitarian")
print(f"CrisisBench splits: {list(crisisbench.keys())}")

# Combine all splits
dfs = []
for split_name in crisisbench:
    df = crisisbench[split_name].to_pandas()
    df["split"] = split_name
    dfs.append(df)

cb_df = pd.concat(dfs, ignore_index=True)
print(f"Total rows: {len(cb_df):,}")
print(f"Columns: {list(cb_df.columns)}")
cb_df.head()

CrisisBench splits: ['test', 'dev', 'train']
Total rows: 141,533
Columns: ['id', 'event', 'source', 'text', 'lang', 'lang_conf', 'class_label', 'split']


,id,event,source,text,lang,lang_conf,class_label,split
0,21767,disaster_events,drd-figureeight-multimedia,Staff at our feeding centre say chronic malnou...,en,1,requests_or_needs,test
1,324921639701716993,2013_west_texas,crisislext6,@johnnyjeff05 you comin down for the summer se...,en,1,not_humanitarian,test
2,262348997677686785,2012_sandy_hurricane-ontopic,crisislext6,@Nooram_M yea it's upstate I'm like a few hour...,en,1,not_humanitarian,test
3,508341453676756992,2014_pakistan_floods,crisisnlp-cf,RT @GDLanglands: Teach every Pakistani that it...,en,1,not_humanitarian,test
4,540037166374866944,2014_philippines_typhoon,crisisnlp-cf,RT @TedWinnerCNN: Stay with @CNN for live cvg ...,en,1,displaced_and_evacuations,test


In [7]:
# Explore CrisisBench structure
print("Language distribution (top 10):")
if "lang" in cb_df.columns:
    print(cb_df["lang"].value_counts().head(10))
else:
    print("No 'lang' column found — checking other columns...")
    print(cb_df.dtypes)

print(f"\nLabel distribution:")
print(cb_df["class_label"].value_counts())

print(f"\nEvents by tweet count (top 20):")
if "event" in cb_df.columns:
    print(cb_df["event"].value_counts().head(20))

Language distribution (top 10):
lang
en    132619
es      4879
it      1337
fr       799
tl       745
pt       527
ru       123
hi        52
id        52
co        36
Name: count, dtype: int64

Label distribution:
class_label
not_humanitarian                       52939
other_relevant_information             44626
donation_and_volunteering               8350
requests_or_needs                       7004
sympathy_and_support                    6829
infrastructure_and_utilities_damage     5496
affected_individual                     4211
caution_and_advice                      3601
injured_or_dead_people                  3168
disease_related                         1478
response_efforts                        1114
personal_update                         1046
missing_and_found_people                 572
displaced_and_evacuations                545
physical_landslide                       538
terrorism_related                         16
Name: count, dtype: int64

Events by tweet count (top 

In [8]:
# Save raw CrisisBench
cb_df.to_parquet(f"{PROJECT_ROOT}/data/raw/crisisbench_raw.parquet", index=False)
print(f"Saved: {PROJECT_ROOT}/data/raw/crisisbench_raw.parquet ({len(cb_df):,} rows)")

Saved: /Users/cyril/Desktop/Un-supervised/data/raw/crisisbench_raw.parquet (141,533 rows)


## 1.3 Summary

Both datasets downloaded and saved as parquet files in `data/raw/`. 

**Next step:** Run notebook `02_preprocessing.py` to clean tweets, remove duplicates, normalize hashtags, and filter CrisisBench to English-only humanitarian tweets.